In [ ]:


import pandas as pd
import numpy as np
import torch
import torch.nn as nn   

In [ ]:
df = pd.read_csv('nyc_311.csv')
print(df.shape)
df.head()

In [ ]:
data = df.copy()

data["created_date"] = pd.to_datetime(data["created_date"]) 
data["closed_date"] = pd.to_datetime(data["closed_date"]) 

data = data.dropna(subset=["closed_date"]) 

data["hours"] = (data["closed_date"] - data["created_date"]).dt.total_seconds() / 3600

data["hour"] = data["created_date"].dt.hour

In [ ]:
X = data[["agency","complaint_type","hour"]] 
Y = data["hours"] 
print(X.shape, Y.shape)

In [ ]:
X_train = X.sample(frac=0.8, random_state=42) 
X_test = X.drop(X_train.index) 

Y_train = Y[X_train.index]
Y_test = Y[X_test.index] 

print(X_train.shape, Y_train.shape)

In [ ]:
# Fill missing values using only the training data
agency_mode = X_train["agency"].mode()[0]
complaint_mode = X_train["complaint_type"].mode()[0]
hour_median = X_train["hour"].median()

X_train = X_train.fillna({"agency": agency_mode, "complaint_type": complaint_mode, "hour": hour_median})
X_test = X_test.fillna({"agency": agency_mode, "complaint_type": complaint_mode, "hour": hour_median})

In [ ]:
X_train = pd.get_dummies(X_train, columns=["agency","complaint_type"],dtype=np.float32)
X_test = pd.get_dummies(X_test, columns=["agency","complaint_type"], dtype=np.float32) 

X_test = X_test.reindex(columns=X_train.columns, fill_value=0) 

print(X_train.shape, X_test.shape)

In [ ]:
# new names (_t) so this cell can be run again safely
X_train_t = torch.tensor(X_train.values, dtype=torch.float32)
Y_train_t = torch.tensor(Y_train.values, dtype=torch.float32).reshape(-1, 1)

X_test_t = torch.tensor(X_test.values, dtype=torch.float32)
Y_test_t = torch.tensor(Y_test.values, dtype=torch.float32).reshape(-1, 1)

print(X_train_t.shape, Y_train_t.shape)
print(X_test_t.shape, Y_test_t.shape)

In [ ]:
print(df.shape)                  # (5000, 6)
print(df["created_date"].dtype)

In [ ]:
from torch.utils.data import TensorDataset

In [ ]:
train_ds = TensorDataset(X_train_t, Y_train_t)

train_ds[0:3]

In [ ]:
from torch.utils.data import DataLoader

batch_size = 500 

train_dl = DataLoader(train_ds, batch_size, shuffle=True)

In [ ]:
X_train_t

In [ ]:
for xb,yb in train_dl:
    print(xb)
    print(yb)
    break

In [ ]:
# inputs = number of columns in X_train_t (after one-hot), outputs = 1 (hours)
model = nn.Linear(X_train_t.shape[1], Y_train_t.shape[1])
print(model.weight)

print(model.bias)

In [ ]:
pred = model(X_train_t)

print(pred)
print(pred.shape)